# RF-DETR: Model Conversion to ONNX & Performance Benchmark
### Export Fine-Tuned PyTorch Weights (.pth) to High-Performance ONNX Format

This notebook converts a trained or fine-tuned **RF-DETR** object detection model to **ONNX** format for deployment on scalable microservice runtimes (such as Azure Kubernetes Service).

---

### Key Capabilities
1. **Auto-Discovery**: Automatically locates trained checkpoints (`.pth`) in the workspace.
2. **Dual Exporter Support**:
   - Native `rfdetr` `.export(format="onnx")`
   - Custom `torch.onnx.export` wrapper with post-processing (Sigmoid probabilities + dynamic batch axes).
3. **Graph Validation**: Runs structural graph checks with `onnx.checker`.
4. **Benchmarking**: Measures ONNX Runtime inference latency (mean ms, p95 ms, and FPS).
5. **Deployment Ready**: Saves the model artifact to `models/rfdetr_model.onnx` and syncs to `server/rfdetr_model.onnx` for immediate container packaging.

In [ ]:
# STEP 0: Install Dependencies
import sys

# Essential ONNX, PyTorch, Simplifier and Runtime dependencies
%pip install -q torch onnx onnxruntime onnx-simplifier onnxscript onnxsim

# OCR & visual analysis dependencies
%pip install -q "paddlepaddle>=2.6.0" "paddleocr>=2.7.0" "opencv-python-headless>=4.9.0.80" "pillow>=10.0.0" "numpy>=1.24.0" "requests>=2.28.0" "matplotlib>=3.7.0"

# Install rfdetr with ONNX and ONNXExport capabilities
try:
    import rfdetr
    print("✓ rfdetr is already installed.")
except ImportError:
    print("Installing rfdetr with onnxexport support...")
    %pip install -q "rfdetr[onnx]" "rfdetr[onnxexport]"

print("✓ Dependencies configured successfully.")


In [ ]:
# CELL 1: Imports & Diagnostic Verifications
import os
import sys
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

import onnx
import onnxruntime as ort

print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__}")
print(f"ONNX Version:         {onnx.__version__}")
print(f"ONNX Runtime Version: {ort.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:           {torch.cuda.get_device_name(0)}")

In [ ]:
# CELL 2: Configuration & Checkpoint Search Paths

# Model architecture & input resolution
MODEL_FAMILY = "RFDETRBase"     # Options: RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
RESOLUTION = 1008               # Target image resolution (1008x1008 or 640x640)
CLASSES = ["blue_aisle", "blue_bay", "location_tag"]

# Candidate paths for trained checkpoints
CANDIDATE_CKPTS = [
    "../multi_class_train_rfdetr/model/best_model_full_data.pth",
    "../multi_class_train_rfdetr/model/best_model_sample_1000.pth",
    "../multi_class_train_rfdetr/runs/rf_detr_full_data/checkpoints/best_loss.pth",
    "../location_tag_single_class_train_rfdetr/model/best_model_full_data.pth",
    "../best_model_full_data.pth",
    "../best_model.pth",
    "best_model.pth"
]

OUTPUT_MODELS_DIR = Path("models")
SERVER_DIR = Path("server")
OUTPUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)
SERVER_DIR.mkdir(parents=True, exist_ok=True)

TARGET_ONNX_PATH = OUTPUT_MODELS_DIR / "rfdetr_model.onnx"

print(f"Target ONNX Output: {TARGET_ONNX_PATH.resolve()}")

In [ ]:
# CELL 3: Discover and Load RF-DETR Model Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

active_ckpt_path = None
for cp in CANDIDATE_CKPTS:
    p = Path(cp)
    if p.exists() and p.stat().st_size > 1024 * 1024:
        active_ckpt_path = str(p.resolve())
        break

print(f"Active Checkpoint: {active_ckpt_path if active_ckpt_path else 'None found (initializing base architecture)'}")

def load_rfdetr_model(model_name: str, num_classes: int, resolution: int):
    try:
        from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
        cls_map = {
            "rfdetrnano": RFDETRNano,
            "rfdetrsmall": RFDETRSmall,
            "rfdetrmedium": RFDETRMedium,
            "rfdetrbase": RFDETRBase,
            "rfdetrlarge": RFDETRLarge
        }
        ModelClass = cls_map.get(model_name.lower(), RFDETRBase)
        
        # Suppress automatic internet downloads
        if hasattr(ModelClass, "maybe_download_pretrain_weights"):
            ModelClass.maybe_download_pretrain_weights = lambda self: None
        if hasattr(ModelClass, "load_pretrain_weights"):
            ModelClass.load_pretrain_weights = lambda self: None
            
        wrapper = ModelClass(num_classes=num_classes, resolution=resolution, pretrain_weights=None)
        
        if hasattr(wrapper, "model") and hasattr(wrapper.model, "model") and isinstance(wrapper.model.model, torch.nn.Module):
            model_core = wrapper.model.model
        elif hasattr(wrapper, "model") and isinstance(wrapper.model, torch.nn.Module):
            model_core = wrapper.model
        else:
            model_core = wrapper
            
        return wrapper, model_core
    except Exception as e:
        print(f"RF-DETR loader notice: {e}")
        return None, None

wrapper, model = load_rfdetr_model(MODEL_FAMILY, len(CLASSES), RESOLUTION)

if active_ckpt_path and model is not None:
    print(f"Loading checkpoint tensor weights from: {active_ckpt_path}")
    checkpoint = torch.load(active_ckpt_path, map_location="cpu", weights_only=False)
    state = checkpoint.get("model", checkpoint)
    
    # Adapt detection head if checkpoint class count differs
    for k in ["class_embed.weight", "model.class_embed.weight", "module.class_embed.weight"]:
        if k in state:
            ckpt_classes = state[k].shape[0]
            if hasattr(model, "class_embed") and model.class_embed.out_features != ckpt_classes:
                print(f"Adapting detection head to {ckpt_classes} classes...")
                in_f = model.class_embed.in_features
                model.class_embed = nn.Linear(in_f, ckpt_classes)
            break
            
    filtered = {k.replace("model.", "").replace("module.", ""): v for k, v in state.items()}
    cur_state = model.state_dict()
    matched = {k: v for k, v in filtered.items() if k in cur_state and cur_state[k].shape == v.shape}
    model.load_state_dict(matched, strict=False)
    print(f"✓ Loaded {len(matched)} / {len(cur_state)} tensor layers from checkpoint.")

if model is not None:
    model.eval()
    print("✓ Model ready for ONNX export.")

In [ ]:
# CELL 4: Export Model to ONNX

import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import contextmanager

# Try importing RF-DETR utilities
try:
    from rfdetr.util.misc import nested_tensor_from_tensor_list
    print("✓ Imported nested_tensor_from_tensor_list from rfdetr.util.misc")
except ImportError:
    nested_tensor_from_tensor_list = None

# Free Python objects referencing tensors & release cached memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Flushed GPU memory cache (torch.cuda.empty_cache)")

# ==============================================================================
# PATCH 1: Dimension-Aware LayerNorm Patch for Projector & Backbone
# Resolves: 
#   1. SymbolicValueError: Gather / ListConstruct dynamic shape in projector.py:43
#   2. RuntimeError: permute(sparse_coo) dimension mismatch (input.dim() == 3 != 4)
# ==============================================================================
def fixed_layernorm_forward(self, x):
    norm_shape = (self.weight.shape[0],) if hasattr(self, 'weight') else tuple(self.normalized_shape)
    # Only 4D spatial tensors (B, C, H, W) require permute for LayerNorm over channel axis
    if x.dim() == 4:
        x = x.permute(0, 2, 3, 1)
        x = F.layer_norm(x, norm_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
    else:
        # 3D token sequences (B, N, C) or 2D tensors are normalized directly across last dimension
        x = F.layer_norm(x, norm_shape, self.weight, self.bias, self.eps)
    return x

# Apply class-level patch to rfdetr.models.backbone.projector.LayerNorm
try:
    from rfdetr.models.backbone import projector
    if hasattr(projector, "LayerNorm"):
        projector.LayerNorm.forward = fixed_layernorm_forward
        print("✓ Patched rfdetr.models.backbone.projector.LayerNorm.forward (dimension-aware)")
except Exception as e:
    print(f"Projector module patch notice: {e}")

# Recursively patch only Projector / 4D LayerNorm modules in loaded model
def patch_model_layernorms(module):
    count = 0
    for name, child in module.named_children():
        is_projector_ln = (
            "projector" in child.__class__.__module__.lower() or
            "projector" in name.lower() or
            child.__class__.__name__ == "LayerNorm2d" or
            (child.__class__.__name__ == "LayerNorm" and "torch.nn" not in child.__class__.__module__)
        )
        if is_projector_ln and hasattr(child, 'weight') and hasattr(child, 'bias'):
            norm_shape = (child.weight.shape[0],)
            def make_forward(c, ns):
                def patched(inp):
                    if inp.dim() == 4:
                        inp = inp.permute(0, 2, 3, 1)
                        inp = F.layer_norm(inp, ns, c.weight, c.bias, c.eps)
                        inp = inp.permute(0, 3, 1, 2)
                    else:
                        inp = F.layer_norm(inp, ns, c.weight, c.bias, c.eps)
                    return inp
                return patched
            child.forward = make_forward(child, norm_shape)
            count += 1
        else:
            count += patch_model_layernorms(child)
    return count

if 'model' in globals() and model is not None:
    patched_count = patch_model_layernorms(model)
    print(f"✓ Verified & patched {patched_count} Projector LayerNorm modules in loaded model")

# ==============================================================================
# PATCH 2: Disable Anti-Aliased Bicubic for ONNX Opset 17
# Resolves: UnsupportedOperatorError: Exporting the operator 'aten::_upsample_bicubic2d_aa'
# ==============================================================================
if not hasattr(F, '_onnx_orig_interpolate'):
    F._onnx_orig_interpolate = F.interpolate

_true_orig_interpolate = F._onnx_orig_interpolate

def _patched_interpolate(input, size=None, scale_factor=None, mode='nearest',
                         align_corners=None, recompute_scale_factor=None, antialias=False):
    if mode == 'bicubic' and antialias:
        antialias = False
    return _true_orig_interpolate(
        input,
        size=size,
        scale_factor=scale_factor,
        mode=mode,
        align_corners=align_corners,
        recompute_scale_factor=recompute_scale_factor,
        antialias=antialias
    )

F.interpolate = _patched_interpolate
print("✓ F.interpolate patched: antialias dynamically disabled for bicubic mode during ONNX trace.")

# Register symbolic handler in ONNX internal registry only if not already registered
try:
    import torch.onnx.symbolic_helper as symbolic_helper
    import torch.onnx.symbolic_opset11 as sym_opset11
    from torch.onnx._internal.registration import registry
    decorated = symbolic_helper._apply_params("upsample_bicubic2d", 4, "cubic")(sym_opset11._interpolate)
    for opset in range(11, 20):
        if hasattr(registry, "is_registered_op") and not registry.is_registered_op("aten::_upsample_bicubic2d_aa", opset):
            registry.register("aten::_upsample_bicubic2d_aa", opset, decorated)
    print("✓ Registered ONNX symbolic handler for 'aten::_upsample_bicubic2d_aa'")
except Exception:
    pass

class RFDETRONNXExportWrapper(nn.Module):
    """
    PyTorch Wrapper for ONNX Tracing:
    - Input:  images (batch_size, 3, height, width)
    - Output: scores (batch_size, num_queries, num_classes) [after Sigmoid]
              boxes  (batch_size, num_queries, 4)           [normalized (cx, cy, w, h)]
    """
    def __init__(self, core_model: nn.Module):
        super().__init__()
        self.model = core_model

    def forward(self, images: torch.Tensor):
        out = self.model(images)
        if isinstance(out, dict):
            pred_logits = out["pred_logits"]
            pred_boxes = out["pred_boxes"]
        elif isinstance(out, (list, tuple)):
            pred_logits, pred_boxes = out[0], out[1]
        else:
            raise ValueError(f"Unexpected output format: {type(out)}")
            
        scores = torch.sigmoid(pred_logits)
        return scores, pred_boxes

def export_onnx(model_to_export, wrapper_obj, out_path: Path, resolution: int):
    print(f"Exporting RF-DETR to ONNX at {out_path} (Resolution: {resolution}x{resolution})...")
    
    # Ensure patches are active on the target model
    patch_model_layernorms(model_to_export)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    exported = False
    if wrapper_obj is not None and hasattr(wrapper_obj, "export"):
        try:
            print("Attempting export with native wrapper.export()...")
            wrapper_obj.export(format="onnx", output_dir=str(out_path.parent))
            for f in out_path.parent.glob("*.onnx"):
                if f.resolve() != out_path.resolve():
                    shutil.move(str(f), str(out_path))
            exported = True
            print("✓ Native export completed.")
        except Exception as e:
            print(f"Native export fallback: {e}")

    if not exported:
        if model_to_export is None:
            raise RuntimeError("Model is not initialized.")
            
        export_wrapper = RFDETRONNXExportWrapper(model_to_export).cpu().eval()
        dummy_input = torch.randn(1, 3, resolution, resolution, dtype=torch.float32)
        
        input_names = ["images"]
        output_names = ["scores", "boxes"]
        dynamic_axes = {
            "images": {0: "batch_size"},
            "scores": {0: "batch_size"},
            "boxes":  {0: "batch_size"}
        }
        
        print("Tracing with torch.onnx.export (opset_version=17)...")
        with torch.no_grad():
            torch.onnx.export(
                export_wrapper,
                dummy_input,
                str(out_path),
                export_params=True,
                opset_version=17,
                do_constant_folding=True,
                input_names=input_names,
                output_names=output_names,
                dynamic_axes=dynamic_axes
            )
        print("✓ PyTorch ONNX export completed.")

    # Optional ONNX Simplification if onnxsim is installed
    try:
        import onnxsim
        print("Simplifying ONNX model with onnxsim...")
        sim_model, check = onnxsim.simplify(str(out_path))
        if check:
            onnx.save(sim_model, str(out_path))
            print("✓ ONNX model successfully simplified!")
    except Exception as e:
        print(f"Notice onnxsim: {e}")

    # Validate ONNX model
    onnx_proto = onnx.load(str(out_path))
    onnx.checker.check_model(onnx_proto)
    size_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f"\n=======================================================")
    print(f"✓ ONNX EXPORT SUCCESSFUL: {out_path.name}")
    print(f"   - File:    {out_path.resolve()}")
    print(f"   - Size:    {size_mb:.2f} MB")
    print(f"   - Inputs:  {[inp.name for inp in onnx_proto.graph.input]}")
    print(f"   - Outputs: {[out.name for out in onnx_proto.graph.output]}")
    print(f"=======================================================\n")

if model is not None:
    export_onnx(model, wrapper, TARGET_ONNX_PATH, RESOLUTION)
    # Sync to server directory for immediate container building
    shutil.copy(TARGET_ONNX_PATH, SERVER_DIR / "rfdetr_model.onnx")
    print(f"✓ Synced model to server directory: {SERVER_DIR / 'rfdetr_model.onnx'}")


In [ ]:
# CELL 5: ONNX Runtime Inference & Latency Benchmark

if not TARGET_ONNX_PATH.exists():
    print(f"Model file not found at: {TARGET_ONNX_PATH}")
else:
    print(f"Creating ONNX Runtime session for: {TARGET_ONNX_PATH}")
    
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if torch.cuda.is_available() else ["CPUExecutionProvider"]
    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.intra_op_num_threads = os.cpu_count() or 4
    
    session = ort.InferenceSession(str(TARGET_ONNX_PATH), sess_options=opts, providers=providers)
    print(f"✓ Session Provider: {session.get_providers()[0]}")
    
    inp_name = session.get_inputs()[0].name
    out_names = [o.name for o in session.get_outputs()]
    
    # Warmup
    test_tensor = np.random.randn(1, 3, RESOLUTION, RESOLUTION).astype(np.float32)
    for _ in range(3):
        _ = session.run(out_names, {inp_name: test_tensor})
        
    # Latency Benchmark
    NUM_ITERS = 20
    latencies = []
    for _ in range(NUM_ITERS):
        t0 = time.perf_counter()
        outs = session.run(out_names, {inp_name: test_tensor})
        latencies.append((time.perf_counter() - t0) * 1000)
        
    avg_ms = np.mean(latencies)
    p95_ms = np.percentile(latencies, 95)
    fps = 1000.0 / avg_ms
    
    print("\n=======================================================")
    print(" ONNX RUNTIME LATENCY BENCHMARK RESULTS")
    print("=======================================================")
    print(f" Iterations:      {NUM_ITERS}")
    print(f" Average Latency: {avg_ms:.2f} ms")
    print(f" P95 Latency:     {p95_ms:.2f} ms")
    print(f" Throughput:      {fps:.1f} FPS")
    print(f" Output 0 Shape:  {outs[0].shape} (Scores/Probabilities)")
    print(f" Output 1 Shape:  {outs[1].shape} (Bounding Boxes)")
    print("=======================================================\n")
    print("Ready for deployment! Next step: run deploy.sh or deploy.py")

In [ ]:
# CELL 8: PaddleOCR Engine Smoke Test & Text Extraction Verification
# Validates the PaddleOCR engine for the two-stage detection + recognition pipeline

try:
    import logging
    logging.getLogger("ppocr").setLevel(logging.WARNING)
    from paddleocr import PaddleOCR
    from PIL import Image, ImageDraw
    
    print("Initializing PaddleOCR engine (English, angle classification)... ")
    use_gpu = torch.cuda.is_available()
    try:
        ocr = PaddleOCR(use_angle_cls=True, lang="en", use_gpu=use_gpu)
    except TypeError:
        ocr = PaddleOCR(use_angle_cls=True, lang="en")
    
    # Create synthetic test crop with sample warehouse location tag text
    sample_crop = Image.new("RGB", (300, 100), color=(255, 255, 255))
    d = ImageDraw.Draw(sample_crop)
    d.text((25, 35), "LOC-A-12-04", fill=(0, 0, 0))
    
    crop_np = np.array(sample_crop)
    ocr_result = ocr.ocr(crop_np, cls=True)
    
    print("=======================================================")
    print("✓ PADDLEOCR ENGINE VALIDATION SUCCESSFUL")
    print("=======================================================")
    if ocr_result and ocr_result[0]:
        for line in ocr_result[0]:
            txt = line[1][0]
            conf = line[1][1]
            print(f"  Detected Text: {txt} (Confidence: {conf:.2f})")
    print("PaddleOCR is ready and bundled into the deployment container!")
except Exception as e:
    print(f"Notice during local PaddleOCR test: {e}")
